In [74]:
import pandas as pd
import numpy as np

In [75]:
data = pd.read_csv('dataset.csv')
data.head()

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,232,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8,Neutral
1,564,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6,Positive
2,788,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,Neutral
3,686,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3,Negative
4,608,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4,Negative


In [76]:
data.isnull().sum()

Student_ID                      0
Age                             0
Gender                          0
Academic_Level                  0
Country                         0
Avg_Daily_Usage_Hours           0
Most_Used_Platform              0
Affects_Academic_Performance    0
Sleep_Hours_Per_Night           0
Mental_Health_Score             0
Overall_Impact                  0
dtype: int64

In [77]:
unique = data['Overall_Impact'].unique()
unique

array(['Neutral', 'Positive', 'Negative'], dtype=object)

In [78]:
balance = data['Overall_Impact'].value_counts()
balance

Overall_Impact
Negative    939
Positive    499
Neutral     267
Name: count, dtype: int64

In [79]:
data = data.drop(columns=['Student_ID'])
data.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8,Neutral
1,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6,Positive
2,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,Neutral
3,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3,Negative
4,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4,Negative


In [80]:
targer_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
data['Overall_Impact'] = data['Overall_Impact'].map(targer_map)

y = data['Overall_Impact']

data = data.drop(columns=['Overall_Impact'])
data.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score
0,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8
1,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6
2,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0
3,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3
4,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4


In [81]:
affect_map = {'No': 0, 'Yes': 1}

data['Affects_Academic_Performance'] = data['Affects_Academic_Performance'].map(affect_map)

data.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score
0,21,Male,Undergraduate,Other,4.0,Facebook,0,6.7,6.8
1,23,Female,Undergraduate,Other,1.6,LinkedIn,0,8.6,7.6
2,22,Male,Graduate,Canada,4.6,Instagram,0,6.7,7.0
3,18,Male,Undergraduate,Other,7.0,Snapchat,1,5.4,5.3
4,24,Female,High School,Other,7.5,Facebook,1,5.0,4.4


In [82]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

onehotcolumns = ['Academic_Level', 'Gender']

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown='ignore'), onehotcolumns)
    ],
    remainder="passthrough"
)

encoder_data = preprocessor.fit_transform(data)
encoder_data

array([[0.0, 0.0, 1.0, ..., 0, 6.7, 6.8],
       [0.0, 0.0, 1.0, ..., 0, 8.6, 7.6],
       [1.0, 0.0, 0.0, ..., 0, 6.7, 7.0],
       ...,
       [0.0, 0.0, 1.0, ..., 1, 6.7, 6.0],
       [1.0, 0.0, 0.0, ..., 0, 7.5, 8.0],
       [0.0, 0.0, 1.0, ..., 1, 6.3, 5.0]], dtype=object)

In [83]:
column_names = preprocessor.get_feature_names_out()
column_names

array(['onehot__Academic_Level_Graduate',
       'onehot__Academic_Level_High School',
       'onehot__Academic_Level_Undergraduate', 'onehot__Gender_Female',
       'onehot__Gender_Male', 'remainder__Age', 'remainder__Country',
       'remainder__Avg_Daily_Usage_Hours',
       'remainder__Most_Used_Platform',
       'remainder__Affects_Academic_Performance',
       'remainder__Sleep_Hours_Per_Night',
       'remainder__Mental_Health_Score'], dtype=object)

In [84]:
data_new = pd.DataFrame(encoder_data, columns=column_names)
data_new.head()

,onehot__Academic_Level_Graduate,onehot__Academic_Level_High School,onehot__Academic_Level_Undergraduate,onehot__Gender_Female,onehot__Gender_Male,remainder__Age,remainder__Country,remainder__Avg_Daily_Usage_Hours,remainder__Most_Used_Platform,remainder__Affects_Academic_Performance,remainder__Sleep_Hours_Per_Night,remainder__Mental_Health_Score
0,0.0,0.0,1.0,0.0,1.0,21,Other,4.0,Facebook,0,6.7,6.8
1,0.0,0.0,1.0,1.0,0.0,23,Other,1.6,LinkedIn,0,8.6,7.6
2,1.0,0.0,0.0,0.0,1.0,22,Canada,4.6,Instagram,0,6.7,7.0
3,0.0,0.0,1.0,0.0,1.0,18,Other,7.0,Snapchat,1,5.4,5.3
4,0.0,1.0,0.0,1.0,0.0,24,Other,7.5,Facebook,1,5.0,4.4


In [85]:
country_freq = data_new['remainder__Country'].value_counts(normalize=True)
data_new['remainder__Country'] = data_new['remainder__Country'].map(country_freq)
data_new.head()

,onehot__Academic_Level_Graduate,onehot__Academic_Level_High School,onehot__Academic_Level_Undergraduate,onehot__Gender_Female,onehot__Gender_Male,remainder__Age,remainder__Country,remainder__Avg_Daily_Usage_Hours,remainder__Most_Used_Platform,remainder__Affects_Academic_Performance,remainder__Sleep_Hours_Per_Night,remainder__Mental_Health_Score
0,0.0,0.0,1.0,0.0,1.0,21,0.391202,4.0,Facebook,0,6.7,6.8
1,0.0,0.0,1.0,1.0,0.0,23,0.391202,1.6,LinkedIn,0,8.6,7.6
2,1.0,0.0,0.0,0.0,1.0,22,0.045748,4.6,Instagram,0,6.7,7.0
3,0.0,0.0,1.0,0.0,1.0,18,0.391202,7.0,Snapchat,1,5.4,5.3
4,0.0,1.0,0.0,1.0,0.0,24,0.391202,7.5,Facebook,1,5.0,4.4


In [86]:
platform_freq = data_new['remainder__Most_Used_Platform'].value_counts(normalize=True)
data_new['remainder__Most_Used_Platform'] = data_new['remainder__Most_Used_Platform'].map(platform_freq)
data_new.head()

,onehot__Academic_Level_Graduate,onehot__Academic_Level_High School,onehot__Academic_Level_Undergraduate,onehot__Gender_Female,onehot__Gender_Male,remainder__Age,remainder__Country,remainder__Avg_Daily_Usage_Hours,remainder__Most_Used_Platform,remainder__Affects_Academic_Performance,remainder__Sleep_Hours_Per_Night,remainder__Mental_Health_Score
0,0.0,0.0,1.0,0.0,1.0,21,0.391202,4.0,0.150147,0,6.7,6.8
1,0.0,0.0,1.0,1.0,0.0,23,0.391202,1.6,0.103226,0,8.6,7.6
2,1.0,0.0,0.0,0.0,1.0,22,0.045748,4.6,0.228152,0,6.7,7.0
3,0.0,0.0,1.0,0.0,1.0,18,0.391202,7.0,0.087390,1,5.4,5.3
4,0.0,1.0,0.0,1.0,0.0,24,0.391202,7.5,0.150147,1,5.0,4.4


In [89]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data_new, y, test_size=0.2, random_state=42)

In [140]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(
    loss='log_loss', #кросс энтропия + soft_max (мультикласс)
    penalty='l2',
    alpha=0.001,
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
model

SGDClassifier(alpha=0.001, class_weight='balanced', loss='log_loss',
              random_state=42)

In [141]:
model.fit(X_train, y_train)

SGDClassifier(alpha=0.001, class_weight='balanced', loss='log_loss',
              random_state=42)

In [142]:
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

y_pred = model.predict(X_test)

print('Accurary:', accuracy_score(y_test, y_pred))
print('Matrix: \n', confusion_matrix(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred, average='macro'))

Accurary: 0.906158357771261
Matrix: 
 [[189   2   6]
 [ 17  27   7]
 [  0   0  93]]
Recall:  0.8296008758833482
